# Lesson 5.1 — Prometheus Custom Metrics & Exporters

**Level: Intermediate**

## Learning Objectives
- Understand the four Prometheus metric types: Counter, Gauge, Histogram, Summary
- Use `prometheus_client` to define metrics with labels
- Build a custom Prometheus exporter that exposes a `/metrics` HTTP endpoint
- Understand label cardinality constraints
- Stream live metrics to a Prometheus scrape target

**Produces:** `prometheus_exporter.py` — a standalone metrics exporter.

*Official refs:*
- *[Prometheus Python Client](https://github.com/prometheus/client_python)*
- *[Prometheus Data Model](https://prometheus.io/docs/concepts/data_model/)*
- *[Prometheus Metric Types](https://prometheus.io/docs/concepts/metric_types/)*

## Concepts

### Prometheus Pull Model

Prometheus **pulls** (scrapes) metrics from an HTTP endpoint on your service every N seconds:

```
Prometheus Server
    └── every 15s: GET http://my-service:9090/metrics
            └── Returns text/plain in Prometheus exposition format
                    └── Stored in time-series DB → Grafana → alerts
```

Install the Python client:
```bash
pip install prometheus-client
```

### Four Metric Types

| Type | Goes up/down? | Use case | Example |
|------|--------------|----------|---------|
| **Counter** | Only up | Total events | Requests, errors, scans |
| **Gauge** | Up & down | Current state | CPU %, instance count |
| **Histogram** | N/A | Distribution of values | Request duration |
| **Summary** | N/A | Pre-computed quantiles | p50/p95/p99 latency |

### Labels — Multi-Dimensional Metrics

Labels add dimensions to a metric:
```
ops_instances_total{region="us-east-1", state="running"} 5
ops_instances_total{region="eu-west-1", state="running"} 3
ops_instances_total{region="us-east-1", state="stopped"} 1
```

**Label cardinality rule:** Only use labels with a **small, finite** set of values
(region, state, environment). Never use user IDs, request IDs, or timestamps as labels —
each unique label combination creates a new time series in the database.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "prometheus_client", "--quiet"], capture_output=True)

import prometheus_client as prom
from prometheus_client import (
    Counter, Gauge, Histogram, Summary,
    CollectorRegistry, generate_latest,
)
print(f"prometheus_client version: {prom.__version__}")

## Code-Along: Step 1 — Define Metrics

In [ ]:
# Use a custom registry so this notebook doesn't collide with the default
registry = CollectorRegistry()

# COUNTER: total scans — monotonically increasing
ops_scans_total = Counter(
    name="ops_scans_total",
    documentation="Total number of resource inventory scans performed.",
    labelnames=["provider", "region", "status"],
    registry=registry,
)

# GAUGE: current instance count — goes up and down
ops_instances_total = Gauge(
    name="ops_instances_total",
    documentation="Current number of cloud instances by state.",
    labelnames=["provider", "region", "state"],
    registry=registry,
)

ops_stopped_instances = Gauge(
    name="ops_stopped_instances_count",
    documentation="Number of stopped/terminated instances (potential cost waste).",
    labelnames=["provider", "region"],
    registry=registry,
)

ops_compliance_score = Gauge(
    name="ops_compliance_score_percent",
    documentation="CIS compliance score as a percentage (0–100).",
    labelnames=["account", "framework"],
    registry=registry,
)

# HISTOGRAM: scan duration — buckets in seconds
ops_scan_duration = Histogram(
    name="ops_scan_duration_seconds",
    documentation="Time taken to complete a resource scan.",
    labelnames=["provider", "scan_type"],
    buckets=[0.1, 0.5, 1.0, 2.5, 5.0, 10.0, 30.0, 60.0],
    registry=registry,
)

# COUNTER: security findings
ops_security_findings = Counter(
    name="ops_security_findings_total",
    documentation="Total security findings detected by automated scans.",
    labelnames=["tool", "severity"],
    registry=registry,
)

print("Metrics registered:")
for metric in [
    "ops_scans_total", "ops_instances_total",
    "ops_stopped_instances_count", "ops_compliance_score_percent",
    "ops_scan_duration_seconds", "ops_security_findings_total",
]:
    print(f"  {metric}")

## Code-Along: Step 2 — Recording Metric Values

In [ ]:
import time, random

def simulate_inventory_scan(provider: str, region: str) -> dict:
    """Simulate an inventory scan and record Prometheus metrics."""
    start = time.monotonic()

    running = random.randint(3, 10)
    stopped = random.randint(0, 3)

    # Record current state (Gauge)
    ops_instances_total.labels(provider=provider, region=region, state="running").set(running)
    ops_instances_total.labels(provider=provider, region=region, state="stopped").set(stopped)
    ops_stopped_instances.labels(provider=provider, region=region).set(stopped)

    # Increment scan counter
    ops_scans_total.labels(provider=provider, region=region, status="success").inc()

    # Observe scan duration
    duration = time.monotonic() - start + random.uniform(0.05, 1.5)
    ops_scan_duration.labels(provider=provider, scan_type="inventory").observe(duration)

    return {"running": running, "stopped": stopped, "duration_s": round(duration, 3)}

# Record compliance score
ops_compliance_score.labels(account="123456789012", framework="CIS-AWS").set(82.5)

# Simulate security findings (Counter: can only inc, never dec)
ops_security_findings.labels(tool="bandit",    severity="HIGH").inc(2)
ops_security_findings.labels(tool="trivy",     severity="CRITICAL").inc(0)
ops_security_findings.labels(tool="pip-audit", severity="HIGH").inc(1)

# Run scans
for provider, regions in [("aws", ["us-east-1", "eu-west-1"]), ("gcp", ["us-central1"])]:
    for region in regions:
        r = simulate_inventory_scan(provider, region)
        print(f"  {provider}/{region}: running={r['running']}, stopped={r['stopped']}, "
              f"duration={r['duration_s']}s")
print("\nMetrics recorded.")

## Code-Along: Step 3 — View the /metrics Output

In [ ]:
# generate_latest() produces the text that the /metrics endpoint returns
output = generate_latest(registry).decode("utf-8")
lines = output.splitlines()
print(f"Total lines in /metrics output: {len(lines)}")
print()

# Filter out comment lines to see just the values
data_lines = [l for l in lines if l and not l.startswith("#")]
print(f"Data lines (non-comment metric values):")
for line in data_lines[:20]:
    print(f"  {line}")
if len(data_lines) > 20:
    print(f"  ... ({len(data_lines) - 20} more)")

## Code-Along: Step 4 — Write the Prometheus Exporter

In [ ]:
%%writefile prometheus_exporter.py
"""Custom Prometheus exporter for the Ops Automation Platform.

Exposes cloud resource, security, and compliance metrics at /metrics.

Usage:
    python prometheus_exporter.py
    # then: curl http://localhost:9090/metrics

Prometheus scrape config (prometheus.yml):
    scrape_configs:
      - job_name: ops-platform
        static_configs:
          - targets: [localhost:9090]

Reference: https://github.com/prometheus/client_python
"""
import os, time, random
from prometheus_client import (
    Counter, Gauge, Histogram,
    CollectorRegistry, start_http_server, generate_latest,
)

registry = CollectorRegistry()

instances_gauge = Gauge(
    "ops_ec2_instances", "EC2 instance count by state.",
    ["region", "state"], registry=registry,
)
scan_counter = Counter(
    "ops_inventory_scans_total", "Total inventory scans performed.",
    ["provider", "status"], registry=registry,
)
scan_duration = Histogram(
    "ops_scan_duration_seconds", "Duration of inventory scans.",
    ["provider"], buckets=[0.1, 0.5, 1.0, 5.0, 30.0], registry=registry,
)
compliance_score = Gauge(
    "ops_compliance_score", "CIS compliance score (0–100).",
    ["account", "framework"], registry=registry,
)
security_findings = Gauge(
    "ops_security_findings", "Active security findings by severity.",
    ["tool", "severity"], registry=registry,
)


def collect_metrics() -> None:
    """Collect current metrics. Replace stubs with real API calls."""
    with scan_duration.labels(provider="aws").time():
        time.sleep(random.uniform(0.05, 0.5))   # simulate API latency
        instances_gauge.labels(region="us-east-1", state="running").set(random.randint(3, 10))
        instances_gauge.labels(region="us-east-1", state="stopped").set(random.randint(0, 3))
    scan_counter.labels(provider="aws", status="success").inc()
    compliance_score.labels(account="123456789012", framework="CIS-AWS").set(
        random.uniform(75, 95)
    )
    security_findings.labels(tool="bandit",    severity="HIGH").set(random.randint(0, 3))
    security_findings.labels(tool="trivy",     severity="CRITICAL").set(0)


if __name__ == "__main__":
    port     = int(os.environ.get("METRICS_PORT",    "9090"))
    interval = int(os.environ.get("SCRAPE_INTERVAL", "30"))

    start_http_server(port, registry=registry)
    print(f"✓  Prometheus exporter started  →  http://0.0.0.0:{port}/metrics")
    print(f"   Collecting metrics every {interval}s.  Press Ctrl+C to stop.")

    collect_metrics()   # first collection immediately
    while True:
        time.sleep(interval)
        collect_metrics()
        print(f"   metrics refreshed at {time.strftime('%H:%M:%S')}")


## Code-Along: Step 5 — Quick HTTP Test

In [ ]:
import threading, time, urllib.request
from prometheus_client import start_http_server

start_http_server(9092, registry=registry)
time.sleep(0.3)

try:
    with urllib.request.urlopen("http://localhost:9092/metrics", timeout=3) as resp:
        body = resp.read().decode()
    values = [l for l in body.splitlines() if l and not l.startswith("#")]
    print(f"✓  /metrics endpoint is live  ({len(values)} value lines)")
    for line in values[:8]:
        print(f"   {line}")
except Exception as e:
    print(f"Could not reach endpoint: {e}")
    print("Run 'python prometheus_exporter.py' manually to test.")

## Exercise

Add a `ops_last_scan_timestamp_seconds` Gauge metric.

**Your task:**
1. Define `Gauge("ops_last_scan_timestamp_seconds", ..., ["provider"])`
2. In `collect_metrics()`, call `.set(time.time())` after each successful scan
3. In Prometheus/Grafana, the expression `time() - ops_last_scan_timestamp_seconds`
   tells you how many seconds ago the last scan ran — useful for "no-data" alerts

Verify it appears in `generate_latest(registry)`.

In [ ]:
# TODO: define ops_last_scan_timestamp_seconds Gauge
# last_scan_ts = Gauge("ops_last_scan_timestamp_seconds", "...", ["provider"], registry=registry)

# TODO: set it in simulate_inventory_scan()
# last_scan_ts.labels(provider=provider).set(time.time())

# TODO: verify
# print(generate_latest(registry).decode())


## Key Takeaways
- Use **Gauge** for values that go up/down; **Counter** for cumulative totals
- Labels must be **low-cardinality** — region, state, environment (not request IDs)
- `start_http_server(port, registry=registry)` is all you need to expose `/metrics`
- Use `with histogram.time():` to automatically observe the duration of a code block
- The exporter pattern (background collection loop + HTTP endpoint) is the standard

---
**Next:** [Lesson 5.2 — OpenTelemetry Python](lesson-5.2-opentelemetry-python.ipynb)